In [3]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures
from joblib import Parallel, delayed

num_nodes = 4
project = "ucr-ursa-major-lesani-lab"
zone = "us-central1-c"
machine_type = "e2-highcpu-8"
image_name = "tsm-sc-image"  # your custom image
subnet = "default"
gcp_username = "tejas"

# Cleanup any existing instances with same prefix
os.system(f'gcloud compute instances delete --zone={zone} --quiet '
          f'$(gcloud compute instances list --filter="name~\'tsm-sc-\'" --format="value(name)")')

# Create commands list
commands = []

for i in range(num_nodes):
    cmd = f'''
    gcloud compute instances create tsm-sc-{i:03} \
        --project={project} \
        --zone={zone} \
        --machine-type={machine_type} \
        --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
        --can-ip-forward \
        --maintenance-policy=MIGRATE \
        --provisioning-model=STANDARD \
        --service-account=961693926925-compute@developer.gserviceaccount.com \
        --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
        --tags=http-server,https-server \
        --create-disk=auto-delete=yes,boot=yes,image={image_name},mode=rw,size=20,type=pd-balanced \
        --no-shielded-secure-boot \
        --shielded-vtpm \
        --shielded-integrity-monitoring \
        --labels=goog-ec-src=vm_add-gcloud \
        --reservation-affinity=any
    '''
    commands.append(cmd.strip())


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


# Parallel instance creation
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(run_command, cmd) for cmd in commands]
    concurrent.futures.wait(futures)

print("All instances launched.")

# Wait a bit for IPs to propagate
import time
time.sleep(30)

# Get IPs
os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
          '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')

with open('tsm_ips.txt', 'r') as f:
    iplist = [line.strip() for line in f.readlines()]

print("🎯 Instance IPs:", iplist)


ERROR: (gcloud.compute.instances.delete) argument INSTANCE_NAMES [INSTANCE_NAMES ...]: Must be specified.
Usage: gcloud compute instances delete INSTANCE_NAMES [INSTANCE_NAMES ...] [optional flags]
  optional flags may be  --delete-disks | --help | --keep-disks | --zone

For detailed information on this command and its flags, run:
  gcloud compute instances delete --help


Running: gcloud compute instances create tsm-sc-000         --project=ucr-ursa-major-lesani-lab         --zone=us-central1-c         --machine-type=e2-highcpu-8         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=961693926925-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boot=yes,image=tsm-sc-image,mode=rw,size=20,type=pd-balanced         --no-shielded-secure-boot         --shielded-vtpm         --shielded-integrity-monitoring         --labels=goog-e

Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].
Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-000  us-central1-c  e2-highcpu-8               10.128.0.56  34.72.40.109  RUNNING
NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-002  us-central1-c  e2-highcpu-8               10.128.0.75  34.9.191.224  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-003  us-central1-c  e2-highcpu-8               10.128.0.61  35.202.167.149  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-001  us-central1-c  e2-highcpu-8               10.128.0.74  34.27.85.194  RUNNING
All instances launched.
🎯 Instance IPs: ['10.128.0.56', '10.128.0.74', '10.128.0.75', '10.128.0.61']


In [16]:





def git_pull_stellar(i):
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'
    print(command)
    output = os.system(command)
    print(output)

# Execute in parallel like your example
results = Parallel(n_jobs=60)(delayed(git_pull_stellar)(i) for i in range(len(iplist)))
print(results)

From https://github.com/tejas-shivanand-mane/stellar-core
   c76a489..0208a61  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   c76a489..0208a61  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   c76a489..0208a61  main       -> origin/main


Updating c76a489..0208a61
Fast-forward
 SetupGCP.ipynb               | 850 ++++++++++++++++++++++++-------------------
 gcp_setup_stellar_private.sh |  73 ++--
 2 files changed, 507 insertions(+), 416 deletions(-)
Updating c76a489..0208a61
Fast-forward
 SetupGCP.ipynb               | 850 ++++++++++++++++++++++++-------------------
 gcp_setup_stellar_private.sh |  73 ++--
 2 files changed, 507 insertions(+), 416 deletions(-)
Updating c76a489..0208a61
Fast-forward
 SetupGCP.ipynb               | 850 ++++++++++++++++++++++++-------------------
 gcp_setup_stellar_private.sh |  73 ++--
 2 files changed, 507 insertions(+), 416 deletions(-)
Updating c76a489..0208a61
Fast-forward
 SetupGCP.ipynb               | 850 ++++++++++++++++++++++++-------------------
 gcp_setup_stellar_private.sh |  73 ++--
 2 files changed, 507 insertions(+), 416 deletions(-)
[None, None, None, None]


From https://github.com/tejas-shivanand-mane/stellar-core
   c76a489..0208a61  main       -> origin/main


In [17]:
def compile_stellar(i):
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
make -j8;"'
    print(command)
    output = os.system(command)
    print(output)

# Execute in parallel like your example
results = Parallel(n_jobs=20)(delayed(compile_stellar)(i) for i in range(len(iplist)))
print(results)

Exception ignored in: <function ResourceTracker.__del__ at 0x7e496757e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7b8091582020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

make  all-recursive
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make[2]: Entering directory '/home/tejas/stellar-core/lib'
Making all in ../lib/libsodium
make[3]: Entering directory '/home/tejas/stellar-core/lib/libsodium'
Making all in builds
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/builds'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/builds'
Making all in contrib
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/contrib'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/contrib'
Making all in dist-build
make  all-recursive
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make  a

/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)
/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)
/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)
/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)


pandoc -s -w man ./doc/xdrc.1.md -o ./doc/xdrc.1
make[4]: Leaving directory '/home/tejas/stellar-core/lib/xdrpp'
make[3]: Leaving directory '/home/tejas/stellar-core/lib/xdrpp'
make[3]: Entering directory '/home/tejas/stellar-core/lib'
make[3]: Nothing to be done for 'all-am'.
make[3]: Leaving directory '/home/tejas/stellar-core/lib'
make[2]: Leaving directory '/home/tejas/stellar-core/lib'
make[3]: Entering directory '/home/tejas/stellar-core/lib'
make[3]: Nothing to be done for 'all-am'.
make[3]: Leaving directory '/home/tejas/stellar-core/lib'
Making all in src
make[2]: Leaving directory '/home/tejas/stellar-core/lib'
Making all in src
make[3]: Entering directory '/home/tejas/stellar-core/lib'
make[3]: Nothing to be done for 'all-am'.
make[3]: Leaving directory '/home/tejas/stellar-core/lib'
make[2]: Leaving directory '/home/tejas/stellar-core/lib'
Making all in src
make[2]: Entering directory '/home/tejas/stellar-core/src'
make[2]: Entering directory '/home/tejas/stellar-core/src'


In [18]:
def setup_stellar_private(i):
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd /home/tejas; \
mkdir stellar-private; \
cp stellar-core/gcp_setup_stellar_private.sh stellar-private/gcp_setup_stellar_private.sh; \
cd stellar-private; chmod +x gcp_setup_stellar_private.sh; \
./gcp_setup_stellar_private.sh start; \
./gcp_setup_stellar_private.sh;"'
    print(command)
    output = os.system(command)
    print(output)

# Execute in parallel like your example
results = Parallel(n_jobs=20)(delayed(setup_stellar_private)(i) for i in range(len(iplist)))
print(results)

mkdir: stellar-private: File exists
./gcp_setup_stellar_private.sh: line 27: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
mkdir: stellar-private: File exists
./gcp_setup_stellar_private.sh: line 27: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory


Generating seed for node1...
Initializing database for node1...
⚠ Warning: new-db failed for node1, continuing...
Initializing database for node2...
⚠ Warning: new-db failed for node2, continuing...
Initializing database for node3...
⚠ Warning: new-db failed for node3, continuing...
Initializing database for node4...
⚠ Warning: new-db failed for node4, continuing...
✅ 4-node private Stellar network setup complete!
Start the nodes with:
/home/tejas/tejas/stellar-core/src/stellar-core run --conf /home/tejas/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/tejas/stellar-core/src/stellar-core run --conf /home/tejas/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/tejas/stellar-core/src/stellar-core run --conf /home/tejas/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/tejas/stellar-core/src/stellar-core run --conf /home/tejas/tejas/stellar-private/node4/stellar-core.cfg &
Generating seed for node1...
Initializing database for node1...
⚠ Warning: new-db failed

./gcp_setup_stellar_private.sh: line 27: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory
./gcp_setup_stellar_private.sh: line 97: /home/tejas/tejas/stellar-core/src/stellar-core: No such file or directory


Initializing database for node3...
⚠ Warning: new-db failed for node3, continuing...
Initializing database for node4...
⚠ Warning: new-db failed for node4, continuing...
✅ 4-node private Stellar network setup complete!
Start the nodes with:
/home/tejas/tejas/stellar-core/src/stellar-core run --conf /home/tejas/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/tejas/stellar-core/src/stellar-core run --conf /home/tejas/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/tejas/stellar-core/src/stellar-core run --conf /home/tejas/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/tejas/stellar-core/src/stellar-core run --conf /home/tejas/tejas/stellar-private/node4/stellar-core.cfg &
Initializing database for node1...
⚠ Warning: new-db failed for node1, continuing...
Initializing database for node2...
⚠ Warning: new-db failed for node2, continuing...
Initializing database for node3...
⚠ Warning: new-db failed for node3, continuing...
Initializing database for node4.

In [2]:

# # Fetch all existing instance names matching tsm-sc-*
# fetch_cmd = f'''
# gcloud compute instances list \
#     --filter="name~'tsm-sc-'" \
#     --format="value(name)"
# '''
# instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
# instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries

# print("\n➡ Existing instances to delete:", instances_to_delete)

# if instances_to_delete:
#     # Use parallel deletion
#     def delete_instance(instance_name):
#         cmd = f'''
#         gcloud compute instances delete {instance_name} \
#             --zone={zone} \
#             --project={project} \
#             --quiet
#         '''
#         print(f"Deleting: {instance_name}")
#         return subprocess.call(cmd, shell=True)

#     with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
#         futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
#         concurrent.futures.wait(futures)

#     print("🧹 All old tsm-sc-* instances deleted.\n")
# else:
#     print("✔ No previous instances found to delete.\n")



➡ Existing instances to delete: ['tsm-sc-000', 'tsm-sc-001', 'tsm-sc-002', 'tsm-sc-003']
Deleting: tsm-sc-000
Deleting: tsm-sc-001
Deleting: tsm-sc-002
Deleting: tsm-sc-003


Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].


🧹 All old tsm-sc-* instances deleted.



Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].
